In [29]:
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit import transpile
from qiskit.visualization import plot_histogram
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.primitives import BackendSamplerV2
 
# Load the qiskit runtime sampler
from qiskit_ibm_runtime import SamplerV2 as Sampler
import numpy as np
from qiskit import QuantumCircuit

In [30]:
bit_num = 127
qc = QuantumCircuit(bit_num, bit_num)
rng = np.random.default_rng()

In [31]:
#Setting random bits and bases for Alice
abits = np.round(rng.random(bit_num))
abase = np.round(rng.random(bit_num))

#State preparation for Alice
for i in range(bit_num):
    if abits[i] == 0:
        if abase[i] == 1:
            qc.h(i)
    if abits[i] == 1:
        if abase[i] == 0:
            qc.x(i)
        if abase[i] == 1:
            qc.x(i)
            qc.h(i)

In [32]:
#Now setting up bases for Bob
bbase = np.round(rng.random(bit_num))

#state preparation for Bob
for k in range(bit_num):
    if bbase[k] == 1:
        qc.h(k)
    qc.measure(k, k)


In [33]:
service = QiskitRuntimeService(channel="ibm_cloud", token="E96vYFdqJwr3cSGU1lal1me-9TWQ0NOuTvldjJ1KOJoj")
##"ibm_quantum_platform"  → for the free public IBM Quantum service and get a new token
print(service.backends())

qiskit_runtime_service._discover_account:WARNING:2025-10-31 16:17:59,542: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2025-10-31 16:18:05,081: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2025-10-31 16:18:05,083: Loading instance: open-instance, plan: open


[<IBMBackend('ibm_fez')>, <IBMBackend('ibm_brisbane')>, <IBMBackend('ibm_torino')>, <IBMBackend('ibm_marrakesh')>]


In [34]:
backend = service.backend("ibm_brisbane")
print(backend.name)
print("Backend:", backend)
print("Backend type:", type(backend))

qiskit_runtime_service.backends:WARNING:2025-10-31 16:18:07,218: Using instance: open-instance, plan: open


ibm_brisbane
Backend: <IBMBackend('ibm_brisbane')>
Backend type: <class 'qiskit_ibm_runtime.ibm_backend.IBMBackend'>


In [35]:
#Now we do Transpile (“transpiling” refers to the process of converting a high-level quantum circuit into a form that can run efficiently on a specific quantum device or simulator.)
target = backend.target
pm = generate_preset_pass_manager(target=target, optimization_level = 3)
qc_isa = pm.run(qc)

In [36]:
# Load the Runtime primitive and session
sampler = Sampler(mode=backend)
 

In [37]:
#Qiskit patterns step 3: Executing the generation
job = sampler.run([qc_isa], shots=1)
counts = job.result()[0].data.c.get_counts()
countsint = job.result()[0].data.c.get_int_counts()

In [38]:
print(job.backend())
print(job.status())


<IBMBackend('ibm_brisbane')>
DONE


In [39]:
#Step 4: Post-processing
#Here we interpret our results and extract useful information.
#Any qubit with a state that was prepared and measured in the same basis
#should have a deterministic outcome, such that only one measurement is
#necessary. Those qubits with states prepared and measured 
#in different bases (which would have probabilistic outcomes and 
#would require many measurements to interpret) will not be used to 
#build up our one-time pad/key

In [40]:
#Now extracting Bob's Bits
keys = counts.keys()
key = list(keys)[0]
bmeas = list(key)
bmeas_ints = []
for n in range(bit_num):
    bmeas_ints.append(int(bmeas[n]))

#Reverse the order to match our input. 
#Qiskit (like most quantum programming frameworks) uses a little-endian
#ordering for qubits and classical bits which means the first qubit is the
#least significant bit in the output binary string.
#Qiskit returns results as binary strings where the last qubit measured
#is the first bit in the string.

bbits = bmeas_ints[::-1]

In [41]:
#Compare Alice's and Bob's measurement bases and collect usable bits
agoodbits = []
bgoodbits = []
match_count = 0
for n in range(bit_num):
    if abase[n] == bbase[n]:
        agoodbits.append(int(abits[n]))
        bgoodbits.append(bbits[n])
        if int(abits[n]) == bbits[n]:
            match_count += 1

In [42]:
print("Alice's bits = ", agoodbits)
print("Bob's bits = ", bgoodbits)
print("fidelity = ", match_count / len(agoodbits))
print("loss = ", 1 - match_count / len(agoodbits))

Alice's bits =  [1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1]
Bob's bits =  [1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1]
fidelity =  1.0
loss =  0.0
